In [1]:
import pandas as pd
import fitz
import requests
import os
import zipfile
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

In [2]:
path = "../data/mexico"

In [3]:
url = "https://www.gob.mx/salud/documentos/datos-abiertos-bases-historicas-de-enfermedades-transmitidas-por-vector"
os.makedirs(f"{path}/PDFs", exist_ok=True)

response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

for link in soup.find_all("a"):
    href = link.get("href")
    if href and href.endswith(".pdf"):
        pdf_url = href if href.startswith("http") else requests.compat.urljoin(url, href)
        #print("PDF encontrado:", pdf_url)
        pdf_response = requests.get(pdf_url)
        filename = os.path.join(f"{path}/PDFs", pdf_url.split("/")[-1])

        with open(filename, "wb") as f:
            f.write(pdf_response.content)

        print("Descargado:", filename)

Descargado: ../data/mexico/PDFs\Datos_abiertos_historicos_etv_2026.pdf
Descargado: ../data/mexico/PDFs\Datos_abiertos_historicos_etv_2025.pdf
Descargado: ../data/mexico/PDFs\Datos_abiertos_historicos_etv_2024.pdf
Descargado: ../data/mexico/PDFs\Datos_abiertos_historicos_etv_2023.pdf
Descargado: ../data/mexico/PDFs\Datos_abiertos_historicos_etv_2022.pdf
Descargado: ../data/mexico/PDFs\Datos_abiertos_historicos_etv_2021.pdf
Descargado: ../data/mexico/PDFs\Datos_abiertos_historicos_etv_2020.pdf


In [4]:
def getLinks(pdf_path,file ='links.txt'):
  L = []
  with fitz.open(pdf_path) as pdf:
    for num_pag, page in enumerate(pdf):
      links = page.get_links()
      for l in links:    
        L.append(l['uri'])
        
  with open(file,'w') as file:
    for link in L:
      file.write(link+"\n")
  return L

def getZips(output_folder= "data/zips",fileOri = "links.txt",fileReady ="readyL.txt" ):
    os.makedirs(output_folder, exist_ok=True)
    #Para no redescargar archivos
    F = open(fileReady, "a").close() #para que cree el archivo si no existe
    with open(fileReady, "r") as f:
        ready = [l.strip() for l in f.readlines()]

    with open(fileOri, "r") as f:
        links = [l.strip() for l in f.readlines()]

    downloads = [f for f in os.listdir(output_folder) if f.endswith(".zip")]
        
    c,s = 0,0
    #while i < len(links):
    for i in tqdm(range(len(links)), desc='Descargando ZIPs', leave=False):
        link = links[i]
        filename = "_".join(link.split("/")[-1].split("_")[-2:])
        if link in ready and filename in downloads:
            s+=1
            continue
        filename = os.path.join(output_folder, filename)
        response = requests.get(link, stream=True)
        with open(filename, "wb") as file:
            for chunk in response.iter_content(chunk_size=10240):
                file.write(chunk)   
        if link not in ready:
            F = open(fileReady, "a")
            F.write(link+"\n")
            F.close()
        c+=1
        #print(s,c,i)
    #print(f"Descarga completa. {c} archivos descargados. {s} archivos omitidos")

def desZip(input_folder = "data/zips"):
    E = []
    i = 0
    Files = [f for f in os.listdir(input_folder) if f.endswith(".zip")]
    #print(f"Carpeta: {input_folder}")
    path = '/'.join(input_folder.split('/')[:-1]+["noZips"])
    os.makedirs(path, exist_ok=True)
    for i in tqdm(range(len(Files)), desc='Descomprimiendo ZIPs', leave=False):
            file = Files[i]
            zip_path = os.path.join(input_folder, file)
            extract_path = os.path.join(path, file.replace(".zip", ""))
            #print(extract_path)
            #renombraremos los archivos para que queden mejor ordenados por mes
            extract_path = f"{path}/dengue_"+extract_path[-4:-2]+extract_path[-6:-4]+extract_path[-2:]
            try:
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall(extract_path)
                #os.remove(zip_path)
                
            except:
                E.append((extract_path.split("/"))[-1])
                #os.remove(zip_path)
                continue
    
    #os.rmdir(input_folder)
    if len(E):
        print(f"\tHubo {len(E)} errores.")
        print("\tLos archivos con errores son los siguientes:")
        for e in E:print("\t\t"+e)

def standardizeDate(symbol, iter):
  L = []
  for k in iter:
    if symbol in k:
      t = k.split(symbol)
      L.append(f"{t[2]}-{t[1]}-{t[0]}")
    else: L.append(k)
  return L

def csvMerge(input_folder = "data",output_folder = "data",filename="dengue.csv"): 
  D = [input_folder+"/"+d for d in os.listdir(input_folder) if os.path.isdir(input_folder+"/"+d)]
  #print(D)
  val = True
  DF = ''
  for i in tqdm(range(len(D)), desc='Fusionando CSV', leave=False):
    #print(i)
    d = D[i]
    Files = [d+"/"+f for f in os.listdir(d)]
    #print(Files)
    #print(i)
    for f in Files:
      #print(i)
      if f.endswith(".csv"):
        df = pd.read_csv(f)
        df['FECHA_ACTUALIZACION'] = standardizeDate("/",df["FECHA_ACTUALIZACION"])
        df['FECHA_SIGN_SINTOMAS'] = standardizeDate("/",df["FECHA_SIGN_SINTOMAS"])
        #print(i)
        if(val):  
          DF = df
          val = False
        else: DF = pd.concat([DF, df], ignore_index=True)
      os.remove(f)
    os.rmdir(d)
  os.rmdir(input_folder)
  if len(DF):DF.to_csv(output_folder+"/"+filename,index=False)
  #return DF



In [6]:
path = "../data/mexico"
os.makedirs(path, exist_ok=True)
for i in tqdm(range(len(os.listdir(f"{path}/PDFs"))), desc=f'Procesando datos', leave=False):
  #if i != 0:continue
  n = (os.listdir(f"{path}/PDFs"))[i]
  pdf_path = f"{path}/PDFs/"+n
  os.makedirs(f"{path}/ano_202{i}", exist_ok=True)
  pbar = tqdm(range(4),desc=f'Procesando ano 202{i}', leave=False)
  L = getLinks(pdf_path,f'{path}/ano_202{i}/links_202{i}.txt')
  pbar.update(1)
  getZips(f"{path}/ano_202{i}/zips",f"{path}/ano_202{i}/links_202{i}.txt",f"{path}/ano_202{i}/ready_202{i}.txt")
  pbar.update(1)
  desZip(f"{path}/ano_202{i}/zips")
  pbar.update(1)
  csvMerge(f"{path}/ano_202{i}/noZips",f"{path}/ano_202{i}")
  pbar.update(1)
  pbar.close()

Procesando datos:   0%|          | 0/7 [00:00<?, ?it/s]

Procesando ano 2020:   0%|          | 0/4 [00:00<?, ?it/s]

Descargando ZIPs:   0%|          | 0/5 [00:00<?, ?it/s]

Descomprimiendo ZIPs:   0%|          | 0/5 [00:00<?, ?it/s]

Fusionando CSV:   0%|          | 0/5 [00:00<?, ?it/s]

...

Fusionando CSV:   0%|          | 0/48 [00:00<?, ?it/s]

Procesando ano 2026:   0%|          | 0/4 [00:00<?, ?it/s]

Descargando ZIPs:   0%|          | 0/12 [00:00<?, ?it/s]

Descomprimiendo ZIPs:   0%|          | 0/12 [00:00<?, ?it/s]

Fusionando CSV:   0%|          | 0/12 [00:00<?, ?it/s]

In [9]:
for i in range(len(os.listdir(f"{path}/PDFs"))):
  df =pd.read_csv(f"{path}/ano_202{i}/dengue.csv")
  print(f'ano_202{i}')
  print(df.shape)
  df = df.drop_duplicates(subset=['ID_REGISTRO','FECHA_SIGN_SINTOMAS'],keep='last')
  print(df.shape)
  df.to_csv(f"{path}/ano_202{i}/mexico202{i}.csv",index=False)
  if i ==0:DF = df
  else: DF = pd.concat([DF, df], ignore_index=True)
#print(DF.shape)
DF = DF.drop_duplicates(subset=['ID_REGISTRO','FECHA_SIGN_SINTOMAS'],keep='last')
DF.to_csv(f"{path}/dengue_mexico.csv",index=False)

ano_2020
(475703, 28)
(120288, 28)
ano_2021
(819358, 28)
(156863, 28)
ano_2022
(991575, 28)
(95963, 28)
ano_2023
(4128814, 28)
(339239, 28)
ano_2024
(8877931, 28)
(557949, 28)
ano_2025
(2936374, 28)
(145049, 28)
ano_2026
(221002, 28)
(157989, 28)
